# 1. The plain crossover, with no model

Before asking whether a model can improve the golden cross, this notebook asks what the golden cross does on its own.

The numbers come from `scripts/baseline.py`. The rules: a decision made at the close is traded at the next open, each trade costs 0.05 percent, and a stock is tested only on the days it was in the S&P 500.

In [ ]:
import os, sys, json
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("dark_background")
plt.rcParams.update({"figure.facecolor": "black", "axes.facecolor": "black", "axes.edgecolor": "#3a3b3e", "axes.grid": True,
                     "grid.color": "#1c1c1e", "figure.figsize": (11, 4.5), "axes.spines.top": False, "axes.spines.right": False})
WHITE, SILVER, GREY, JADE = "#f2f2f0", "#d4d6da", "#8d8d90", "#5fd0a0"
pd.options.display.float_format = "{:.3f}".format

## The headline numbers

Every index member gets an equal slice of the money. The slice is either always invested (holding) or invested only while the 50 day average is above the 200 day average (the crossover).

In [ ]:
result = json.load(open("results/baseline.json"))
main = result["by_cost"]["0.05% per trade"]
names = {"strategy": "Crossover", "hold_all_members": "Holding the same stocks", "spy": "Holding SPY"}
table = pd.DataFrame({names[k]: v for k, v in main["portfolio"].items()}).T[["yearly_return", "worst_drop", "yearly_swing"]]
table.columns = ["Yearly return", "Worst drop", "Yearly swing"]
table.style.format("{:.1%}")

## Growth of one dollar

The vertical scale is logarithmic, so equal distances mean equal percentage moves.

In [ ]:
growth = pd.read_csv("results/baseline_growth.csv", index_col=0, parse_dates=True)
fig, ax = plt.subplots()
ax.plot(growth["hold_all_members"], color=SILVER, lw=1.3, label="Holding the same stocks")
ax.plot(growth["spy"], color=GREY, lw=1.1, label="Holding SPY")
ax.plot(growth["strategy"], color=JADE, lw=1.6, label="Crossover")
ax.set_yscale("log"); ax.set_ylabel("Value of 1 dollar"); ax.legend(frameon=False)
plt.show()

## How far each fell from its peak

This is where the crossover earns its keep. It steps out during long declines.

In [ ]:
fig, ax = plt.subplots()
for column, color, label in [("hold_all_members", SILVER, "Holding the same stocks"), ("strategy", JADE, "Crossover")]:
    ax.fill_between(growth.index, growth[column] / growth[column].cummax() - 1, 0, color=color, alpha=0.35, lw=0, label=label)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}"); ax.set_ylabel("Below previous peak"); ax.legend(frameon=False, loc="lower right")
plt.show()

## Year by year

The pattern repeats: smaller losses in the bad years, and a slow start in the year after.

In [ ]:
years = pd.DataFrame(main["portfolio_by_year"]).T
years.index = years.index.astype(int)
fig, ax = plt.subplots()
x = np.arange(len(years))
ax.bar(x - 0.2, years["hold_all_members"], 0.4, color=SILVER, label="Holding the same stocks")
ax.bar(x + 0.2, years["strategy"], 0.4, color=JADE, label="Crossover")
ax.set_xticks(x, years.index, rotation=60); ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}"); ax.legend(frameon=False)
plt.show()

## Stock by stock, and trade by trade

In [ ]:
stocks = pd.read_csv("results/baseline_per_stock.csv")
trades = pd.read_csv("results/baseline_trades.csv")
print(f"Stocks tested: {len(stocks)}")
print(f"Crossover beat holding the same stock for {(stocks.strategy_yearly > stocks.hold_yearly).mean():.0%} of them")
print(f"Trades: {len(trades):,}, of which {(trades.result > 0).mean():.0%} made money")
print(f"Average winner {trades.result[trades.result > 0].mean():+.0%}, average loser {trades.result[trades.result <= 0].mean():+.0%}")
fig, ax = plt.subplots()
ax.hist(trades["result"].clip(-0.5, 2), bins=80, color=SILVER)
ax.axvline(0, color=JADE, lw=1); ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_xlabel("Result of one trade, after costs (capped at 200%)"); ax.set_ylabel("Trades")
plt.show()

## What this says

* The crossover made money, and less than holding.
* It cut the worst drop by more than half.
* Most trades lose a little. A few long trends make all the profit.
* It behaves much like keeping about half the money in stocks.

So the open question is the one the original project asked: **can anything tell the winning crossovers from the losing ones in advance?** That is notebook 2.

One limit: price history was available for 613 of the 990 companies that were in the index over these years. The missing ones mostly failed or were bought out, so every line above is likely too high.